In [1]:
# Run everytime a new function is added
import numpy as np
from scripts.utilities import *
from scripts.features import *
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [2]:
consumer_df, account_df, transaction_df = get_data()
transaction_df.amount = transaction_df.amount.apply(abs)

Data successfully loaded and processed.


In [3]:
c_df = consumer_df.dropna(subset='DQ_TARGET')
a_df = account_df[account_df.prism_consumer_id.isin(c_df.prism_consumer_id)]
t_df = transaction_df[transaction_df.prism_consumer_id.isin(c_df.prism_consumer_id)]

## Account balance overtime:

- Balance recorded at the time account_df was made:

In [4]:
balance = a_df.groupby(['prism_consumer_id']).agg({'balance_date':'max', 'balance':'sum'})
display(balance)
acct_balance = balance['balance']
acct_balance

,balance_date,balance
prism_consumer_id,,
0,2021-08-31,320.37
1,2021-06-30,3302.42
2,2021-04-30,2805.36
3,2021-02-28,7667.01
4,2021-09-30,394.55
...,...,...
13995,2022-01-22,1028.80
13996,2022-02-01,11495.77
13997,2021-12-15,2396.85


prism_consumer_id
0          320.37
1         3302.42
2         2805.36
3         7667.01
4          394.55
           ...   
13995     1028.80
13996    11495.77
13997     2396.85
13998    14835.71
13999      -41.00
Name: balance, Length: 10408, dtype: float64

- Current balance:

In [5]:
t = t_df.copy()

In [6]:
t['balance_date'] = balance['balance_date']
t['amount'] = np.where(t['credit_or_debit'] == 'DEBIT', -t['amount'], t['amount'])
t['is_before_balance_date'] = np.where(t['posted_date'] < t['balance_date'], True, False)
update_balance = t[t['is_before_balance_date'] == False]
update_balance = update_balance.groupby(['prism_consumer_id'])['amount'].sum()

In [7]:
current_balance = c_df[['prism_consumer_id']]

current_balance['balance'] = current_balance['prism_consumer_id'].map(acct_balance).fillna(0)
current_balance['update'] = current_balance['prism_consumer_id'].map(update_balance).fillna(0)
current_balance['current_balance'] = current_balance['balance'] + current_balance['update']
current_balance = current_balance.current_balance
current_balance

C:\Users\bdion\AppData\Local\Temp\ipykernel_13900\2693366838.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_balance['balance'] = current_balance['prism_consumer_id'].map(acct_balance).fillna(0)
C:\Users\bdion\AppData\Local\Temp\ipykernel_13900\2693366838.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_balance['update'] = current_balance['prism_consumer_id'].map(update_balance).fillna(0)


0         -201.22
1         5107.85
2         3235.49
3        10462.25
4        -2149.05
           ...   
13995     1873.31
13996    11173.97
13997     2444.61
13998    24322.07
13999    -1230.10
Name: current_balance, Length: 12000, dtype: float64

## Spending overtime:

In [8]:
inflows = t_df[t_df.credit_or_debit == 'CREDIT']
outflows = t_df[t_df.credit_or_debit == 'DEBIT']

# display(inflows, outflows)
inflows.shape, outflows.shape

((878829, 6), (4258005, 6))

In [9]:
avg_spending = outflows.groupby('prism_consumer_id')['amount'].mean()
avg_spending

prism_consumer_id
0         40.293000
1         95.055021
2         60.857166
3         90.209136
4         65.825977
            ...    
13995     47.222222
13996    102.627890
13997    825.034444
13998    213.410467
13999    104.136830
Name: amount, Length: 11377, dtype: float64

In [10]:
outflows['year'] = outflows['posted_date'].dt.year
outflows['month'] = outflows['posted_date'].dt.month
outflows['week'] = outflows['posted_date'].dt.isocalendar().week
monthly_totals = outflows.groupby(['prism_consumer_id', 'year', 'month'])['amount'].sum().groupby('prism_consumer_id').mean()
weekly_totals  = outflows.groupby(['prism_consumer_id', 'year', 'week'])['amount'].sum().groupby('prism_consumer_id').mean()
yearly_totals  = outflows.groupby(['prism_consumer_id', 'year', 'year'])['amount'].sum().groupby('prism_consumer_id').mean()

display(outflows)

C:\Users\bdion\AppData\Local\Temp\ipykernel_13900\2599825155.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outflows['year'] = outflows['posted_date'].dt.year
C:\Users\bdion\AppData\Local\Temp\ipykernel_13900\2599825155.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outflows['month'] = outflows['posted_date'].dt.month
C:\Users\bdion\AppData\Local\Temp\ipykernel_13900\2599825155.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_inde

,prism_consumer_id,prism_transaction_id,amount,credit_or_debit,posted_date,category,year,month,week
41,3023,41,60.00,DEBIT,2021-04-14,SELF_TRANSFER,2021,4,15
42,3023,42,12.38,DEBIT,2021-04-20,EXTERNAL_TRANSFER,2021,4,16
43,3023,43,150.00,DEBIT,2021-05-02,SELF_TRANSFER,2021,5,17
44,3023,44,200.00,DEBIT,2021-05-03,SELF_TRANSFER,2021,5,18
45,3023,45,67.07,DEBIT,2021-08-19,EXTERNAL_TRANSFER,2021,8,33
...,...,...,...,...,...,...,...,...,...
6407316,10533,6405304,4.96,DEBIT,2022-03-11,BILLS_UTILITIES,2022,3,10
6407317,10533,6405305,63.48,DEBIT,2022-03-30,LOAN,2022,3,13
6407318,10533,6405306,53.99,DEBIT,2022-03-30,LOAN,2022,3,13
6407319,10533,6405307,175.98,DEBIT,2022-03-31,LOAN,2022,3,13


- Visualization for balance changes over time:

In [11]:
balance_changes = t[t['is_before_balance_date'] == False].sort_values(['prism_consumer_id', 'posted_date'])
balance_changes

,prism_consumer_id,prism_transaction_id,amount,credit_or_debit,posted_date,category,balance_date,is_before_balance_date
136802,0,136738,-27.62,DEBIT,2021-03-16,FOOD_AND_BEVERAGES,NaN,False
136767,0,136703,1400.00,CREDIT,2021-03-17,TAX,NaN,False
136803,0,136739,-25.10,DEBIT,2021-03-17,FITNESS,NaN,False
136804,0,136740,-500.00,DEBIT,2021-03-17,BANKING_CATCH_ALL,NaN,False
136805,0,136741,-25.00,DEBIT,2021-03-18,FOOD_AND_BEVERAGES,NaN,False
...,...,...,...,...,...,...,...,...
6277366,13999,6275354,-2.00,DEBIT,2022-01-21,SELF_TRANSFER,NaN,False
6276795,13999,6274783,2.00,CREDIT,2022-01-24,SELF_TRANSFER,NaN,False
6277367,13999,6275355,-41.23,DEBIT,2022-01-24,FOOD_AND_BEVERAGES,NaN,False
6277368,13999,6275356,-107.98,DEBIT,2022-01-24,ENTERTAINMENT,NaN,False


In [12]:
balance_changes['amount'] = pd.to_numeric(balance_changes['amount'], errors='coerce')

In [13]:
initial_balance = acct_balance.to_dict()

In [14]:
from collections import defaultdict

changes = defaultdict(list)

for id, amount in balance_changes[['prism_consumer_id', 'amount']].values:
    changes[id].append(amount)

In [15]:
balance_change = defaultdict(list)
for i, v in changes.items():  
    if i in acct_balance.keys():
        v.insert(0, acct_balance[i])  
    updated_balance = np.cumsum(v)
    balance_change[i] = updated_balance

print(len(balance_change))

11601


## Getting stats for balance changes overtime:

In [16]:
balance_change = pd.DataFrame(
    [(k, v) for k, lst in balance_change.items() for v in lst], 
    columns=['prism_consumer_id', 'balance_changes']
)
# balance_change

In [17]:
stats_changes = balance_change.groupby(['prism_consumer_id']).agg({'balance_changes': ['mean', 'std']})
stats_changes

balance_changes             
                             mean          std
prism_consumer_id                             
0.0                    334.377359   986.039607
1.0                   5008.010698  1196.388690
2.0                   5022.325479  2504.559496
3.0                   7278.493493  1939.603669
4.0                   -674.058730   659.354952
...                           ...          ...
13995.0               1539.005873   293.333878
13996.0              12404.749627  2230.389122
13997.0               3371.028372  1236.834973
13998.0              20413.493344  4331.819603
13999.0               -185.747930  1191.204485

[11601 rows x 2 columns]

## Time specific feats for balance:

In [18]:
initial_date = balance['balance_date']
initial_date

prism_consumer_id
0        2021-08-31
1        2021-06-30
2        2021-04-30
3        2021-02-28
4        2021-09-30
            ...    
13995    2022-01-22
13996    2022-02-01
13997    2021-12-15
13998    2022-01-30
13999    2022-01-26
Name: balance_date, Length: 10408, dtype: object

## Required features:

In [19]:
feats = t_df.groupby(['prism_consumer_id', 'category']).agg({'amount': ['count', 'sum', 'std', 'mean', 'median']})
feats = feats.unstack(level=1)
feats.columns = ['_'.join(col).strip() for col in feats.columns.values]
feats = feats.fillna(0)
# feats = feats.reset_index()
feats.head()

,amount_count_ACCOUNT_FEES,amount_count_ATM_CASH,amount_count_AUTOMOTIVE,amount_count_AUTO_LOAN,amount_count_BANKING_CATCH_ALL,amount_count_BILLS_UTILITIES,amount_count_BNPL,amount_count_CHILD_DEPENDENTS,amount_count_CORPORATE_PAYMENTS,amount_count_CREDIT_CARD_PAYMENT,...,amount_median_REFUND,amount_median_RENT,amount_median_RISK_CATCH_ALL,amount_median_RTO_LTO,amount_median_SELF_TRANSFER,amount_median_TAX,amount_median_TIME_OR_STUFF,amount_median_TRANSPORATION,amount_median_TRAVEL,amount_median_UNEMPLOYMENT_BENEFITS
prism_consumer_id,,,,,,,,,,,,,,,,,,,,,
0,0.0,3.0,21.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,...,19.96,0.0,0.0,0.0,36.15,1043.00,0.0,2.480,54.375,0.0
1,0.0,35.0,7.0,0.0,0.0,0.0,14.0,0.0,0.0,0.0,...,2.42,0.0,0.0,0.0,200.00,1162.70,0.0,25.900,0.000,0.0
2,0.0,9.0,44.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,28.24,0.0,0.0,0.0,55.00,3047.24,0.0,12.250,138.400,0.0
3,0.0,2.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,18.94,0.0,0.0,0.0,300.00,600.00,0.0,24.935,0.000,651.0
4,0.0,10.0,29.0,0.0,0.0,10.0,0.0,0.0,0.0,2.0,...,5.42,0.0,0.0,0.0,250.00,0.00,0.0,75.000,97.980,760.0


- Putting everything together:

In [20]:
result = c_df[['prism_consumer_id', 'DQ_TARGET']].drop_duplicates().reset_index(drop=True)

result['current_balance'] = result['prism_consumer_id'].map(current_balance)
result['balance_mean'] = result['prism_consumer_id'].map(stats_changes[('balance_changes', 'mean')])
result['balance_std'] = result['prism_consumer_id'].map(stats_changes[('balance_changes', 'std')])
result['avg_spending'] = result['prism_consumer_id'].map(avg_spending).fillna(0)
result['avg_monthly_outflow'] = result['prism_consumer_id'].map(monthly_totals).fillna(0)
result['avg_weekly_outflow'] = result['prism_consumer_id'].map(weekly_totals).fillna(0)
result['avg_yearly_outflow'] = result['prism_consumer_id'].map(yearly_totals).fillna(0)

mapped_feats = pd.DataFrame({col: result['prism_consumer_id'].map(feats[col]) for col in feats.columns})
result = pd.concat([result, mapped_feats], axis=1)
result

,prism_consumer_id,DQ_TARGET,current_balance,balance_mean,balance_std,avg_spending,avg_monthly_outflow,avg_weekly_outflow,avg_yearly_outflow,amount_count_ACCOUNT_FEES,...,amount_median_REFUND,amount_median_RENT,amount_median_RISK_CATCH_ALL,amount_median_RTO_LTO,amount_median_SELF_TRANSFER,amount_median_TAX,amount_median_TIME_OR_STUFF,amount_median_TRANSPORATION,amount_median_TRAVEL,amount_median_UNEMPLOYMENT_BENEFITS
0,0,0.0,-201.22,334.377359,986.039607,40.293000,2129.772857,573.400385,14908.410,0.0,...,19.960,0.000,0.0,0.0,36.150,1043.00,0.000,2.480,54.375,0.0
1,1,0.0,5107.85,5008.010698,1196.388690,95.055021,3299.767143,855.495185,23098.370,0.0,...,2.420,0.000,0.0,0.0,200.000,1162.70,0.000,25.900,0.000,0.0
2,2,0.0,3235.49,5022.325479,2504.559496,60.857166,3190.654286,797.663571,11167.290,0.0,...,28.240,0.000,0.0,0.0,55.000,3047.24,0.000,12.250,138.400,0.0
3,3,0.0,10462.25,7278.493493,1939.603669,90.209136,2835.144286,708.786071,9923.005,0.0,...,18.940,0.000,0.0,0.0,300.000,600.00,0.000,24.935,0.000,651.0
4,4,0.0,-2149.05,-674.058730,659.354952,65.825977,2501.387143,795.895909,8754.855,0.0,...,5.420,0.000,0.0,0.0,250.000,0.00,0.000,75.000,97.980,760.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,13995,0.0,1873.31,1539.005873,293.333878,47.222222,106.250000,53.125000,425.000,0.0,...,0.000,0.000,0.0,0.0,9.000,0.00,4.345,0.000,0.000,0.0
11996,13996,0.0,11173.97,12404.749627,2230.389122,102.627890,5998.030000,1458.980270,26991.135,11.0,...,42.180,1229.085,0.0,97.9,100.000,250.00,0.000,10.170,258.310,0.0
11997,13997,0.0,2444.61,3371.028372,1236.834973,825.034444,1856.327500,1485.062000,7425.310,0.0,...,0.000,0.000,0.0,0.0,0.945,307.84,0.000,0.000,0.000,0.0
11998,13998,0.0,24322.07,20413.493344,4331.819603,213.410467,5074.426667,1304.852571,22834.920,0.0,...,0.735,0.000,0.0,0.0,300.000,0.00,0.000,0.000,0.000,0.0


In [21]:
result.isna().sum()

prism_consumer_id                        0
DQ_TARGET                                0
current_balance                          0
balance_mean                           399
balance_std                            412
                                      ... 
amount_median_TAX                      399
amount_median_TIME_OR_STUFF            399
amount_median_TRANSPORATION            399
amount_median_TRAVEL                   399
amount_median_UNEMPLOYMENT_BENEFITS    399
Length: 244, dtype: int64

In [22]:
result = result.fillna(0)

## Predicting:

In [23]:
cids = c_df.prism_consumer_id.unique()
print(len(cids))
train_cids, test_cids = train_test_split(cids, test_size=0.3, random_state=420) # 70 / 30 split

# Features and labels for train and test sets:
X_train = result[result.prism_consumer_id.isin(train_cids)]
y_train = X_train.DQ_TARGET
X_test  = result[result.prism_consumer_id.isin(test_cids)]
y_test  = X_test.DQ_TARGET

X_train.drop(columns=['prism_consumer_id', 'DQ_TARGET'], inplace=True)
X_test.drop(columns=['prism_consumer_id', 'DQ_TARGET'], inplace=True)

len(X_train), len(X_test)

12000


C:\Users\bdion\AppData\Local\Temp\ipykernel_13900\2438821525.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train.drop(columns=['prism_consumer_id', 'DQ_TARGET'], inplace=True)
C:\Users\bdion\AppData\Local\Temp\ipykernel_13900\2438821525.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test.drop(columns=['prism_consumer_id', 'DQ_TARGET'], inplace=True)


(8400, 3600)

In [24]:
clf = LogisticRegression(random_state=420, penalty=None).fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.92      1.00      0.96      3303
         1.0       0.21      0.01      0.03       297

    accuracy                           0.91      3600
   macro avg       0.56      0.50      0.49      3600
weighted avg       0.86      0.91      0.88      3600



c:\Users\bdion\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [25]:
y_pred.sum()

19.0

In [26]:
y_test.sum()

297.0

In [27]:
coefficients = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': clf.coef_[0]})

coefficients.sort_values('Coefficient', ascending=False)[:10]

,Feature,Coefficient
135,amount_std_PAYCHECK,0.000392
112,amount_std_DEPOSIT,0.000390
116,amount_std_EXTERNAL_TRANSFER,0.000368
87,amount_sum_OVERDRAFT,0.000326
102,amount_std_ATM_CASH,0.000299
61,amount_sum_CHILD_DEPENDENTS,0.000191
78,amount_sum_HOME_IMPROVEMENT,0.000157
132,amount_std_MORTGAGE,0.000153
139,amount_std_RENT,0.000150
4,avg_monthly_outflow,0.000149


- Positive coefficients → Increase probability of the positive class.
- Negative coefficients → Decrease probability.

In [28]:
clf = LogisticRegression(random_state=420, penalty=None, class_weight='balanced').fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.95      0.72      0.82      3303
         1.0       0.16      0.58      0.25       297

    accuracy                           0.71      3600
   macro avg       0.55      0.65      0.53      3600
weighted avg       0.88      0.71      0.77      3600



c:\Users\bdion\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [29]:
y_pred.sum()

1100.0

In [30]:
y_test.sum()

297.0

In [31]:
coefficients = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': clf.coef_[0]})

coefficients.sort_values('Coefficient', ascending=False)[:10]

,Feature,Coefficient
87,amount_sum_OVERDRAFT,0.000311
135,amount_std_PAYCHECK,0.000306
94,amount_sum_RTO_LTO,0.000271
5,avg_weekly_outflow,0.000228
131,amount_std_MISCELLANEOUS,0.000209
123,amount_std_GROCERIES,0.000197
61,amount_sum_CHILD_DEPENDENTS,0.000182
178,amount_mean_MISCELLANEOUS,0.000163
116,amount_std_EXTERNAL_TRANSFER,0.000150
112,amount_std_DEPOSIT,0.000149
